<a href="https://colab.research.google.com/github/chervov/ProteinModelling1/blob/main/JAK1_figures_small.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install py3Dmol requests

# Download and shows sequeces

In [2]:
!pip -q install biopython requests

import io
import requests
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1

BASE = "https://api.github.com/repos/chervov/ProteinModelling1"
RAW = "https://raw.githubusercontent.com/chervov/ProteinModelling1/main"

response = requests.get(f"{BASE}/git/trees/main?recursive=1")
response.raise_for_status()

files = sorted(
    f["path"] for f in response.json()["tree"]
    if f["path"].startswith("JAK1/")
    and f["path"].lower().endswith(".pdb")
)

parser = PDBParser(QUIET=True)

for path in files:
    response = requests.get(f"{RAW}/{requests.utils.quote(path, safe='/')}")
    response.raise_for_status()
    structure = parser.get_structure(path, io.StringIO(response.text))

    print(f"\nFILE: {path}")
    for chain in structure[0]:
        sequence = "".join(
            seq1(res.resname, custom_map={"MSE": "M"}, undef_code="X")
            for res in chain
            if "CA" in res and (res.id[0] == " " or res.resname == "MSE")
        )
        if sequence:
            print(f"Chain {chain.id}: length = {len(sequence)}")
            print(sequence)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 19.6 MB/s eta 0:00:00

FILE: JAK1/Jeanpierre2025_Closed_WT_JAK1_best_model.pdb
Chain A: length = 1154
MQYLNIKEDCNAMAFCAKMRSSKKTEVNLEAPEPGVEVIFYLSDREPLRLGSGEYTAEELCIRAAQACRISPLCHNLFALYDENTKLWYAPNRTITVDDKMSLRLHYRMRFYFTNWHGTNDNEQSVWRHSPKKQKNGYEKKKIPDATPLLDASSLEYLFAQGQYDLVKCLAPIRDPKTEQDGHDIENECLGMAVLAISHYAMMKKMQLPELPKDISYKRYIPETLNKSIRQRNLLTRMRINNVFKDFLKEFNNKTICDSSVSTHDLKVKYLATLETLTKHYGAEIFETSMLLISSENEMNWFHSNDGGNVLYYEVMVTGNLGIQWRHKPNVVSVEKEKNKLKRKKLENKHKKDEEKNKIREEWNNFSYFPEITHIVIKESVVSINKQDNKKMELKLSSHEEALSFVSLVDGYFRLTADAHHYLCTDVAPPLIVHNIQNGCHGPICTEYAINKLRQEGSEEGMYVLRWSCTDFDNILMTVTCFEKSEQVQGAQKQFKNFQIEVQKGRYSLHGSDRSFPSLGDLMSHLKKQILRTDNISFMLKRCCQPKPREISNLLVATKKAQEWQPVYPMSQLSFDRILKKDLVQGEHLGRGTRTHIYSGTLMDYKDDEGTSEEKKIKVILKVLDPSHRDISLAFFEAASMMRQVSHKHIVYLYGVCVRDVENIMVEEFVEGGPLDLFMHRKSDVLTTPWKFKVAKQLASALSYLEDKDLVHGNVCTKNLLLAREGIDSECGPFIKLSDPGIPITVLSRQECIERIPWIAPECVEDSKNLSVAADKWSFGTTLWEICYNGEIPLKDKTLIEKERFYESRCRPVTPSCKELADLMTRCMNYDPNQRPFFRAIMR

In [3]:
# Find the current AlphaFold model for human JAK1.
response = requests.get(
    "https://alphafold.ebi.ac.uk/api/prediction/P23458",
    timeout=60,
)
response.raise_for_status()
pdb_url = response.json()[0]["pdbUrl"]

# Download and read the structure.
response = requests.get(pdb_url, timeout=60)
response.raise_for_status()
af2_jak1_pdb = response.text

structure = parser.get_structure("AF_JAK1", io.StringIO(af2_jak1_pdb))

print("AlphaFold human JAK1 — P23458")
print("Model:", pdb_url)

for chain in structure[0]:
    sequence = "".join(
        seq1(res.resname, custom_map={"MSE": "M"}, undef_code="X")
        for res in chain
        if "CA" in res and (res.id[0] == " " or res.resname == "MSE")
    )
    if sequence:
        print(f"\nChain {chain.id}: length = {len(sequence)}")
        print(sequence)

AlphaFold human JAK1 — P23458
Model: https://alphafold.ebi.ac.uk/files/AF-P23458-F1-model_v6.pdb

Chain A: length = 1154
MQYLNIKEDCNAMAFCAKMRSSKKTEVNLEAPEPGVEVIFYLSDREPLRLGSGEYTAEELCIRAAQACRISPLCHNLFALYDENTKLWYAPNRTITVDDKMSLRLHYRMRFYFTNWHGTNDNEQSVWRHSPKKQKNGYEKKKIPDATPLLDASSLEYLFAQGQYDLVKCLAPIRDPKTEQDGHDIENECLGMAVLAISHYAMMKKMQLPELPKDISYKRYIPETLNKSIRQRNLLTRMRINNVFKDFLKEFNNKTICDSSVSTHDLKVKYLATLETLTKHYGAEIFETSMLLISSENEMNWFHSNDGGNVLYYEVMVTGNLGIQWRHKPNVVSVEKEKNKLKRKKLENKHKKDEEKNKIREEWNNFSYFPEITHIVIKESVVSINKQDNKKMELKLSSHEEALSFVSLVDGYFRLTADAHHYLCTDVAPPLIVHNIQNGCHGPICTEYAINKLRQEGSEEGMYVLRWSCTDFDNILMTVTCFEKSEQVQGAQKQFKNFQIEVQKGRYSLHGSDRSFPSLGDLMSHLKKQILRTDNISFMLKRCCQPKPREISNLLVATKKAQEWQPVYPMSQLSFDRILKKDLVQGEHLGRGTRTHIYSGTLMDYKDDEGTSEEKKIKVILKVLDPSHRDISLAFFEAASMMRQVSHKHIVYLYGVCVRDVENIMVEEFVEGGPLDLFMHRKSDVLTTPWKFKVAKQLASALSYLEDKDLVHGNVCTKNLLLAREGIDSECGPFIKLSDPGIPITVLSRQECIERIPWIAPECVEDSKNLSVAADKWSFGTTLWEICYNGEIPLKDKTLIEKERFYESRCRPVTPSCKELADLMTRCMNYDPNQRPFFRAIMRDINKLEEQNPDIVSEKKPATEVDPTHFEKRFLKRIR

# Color by domains

In [5]:
import io
import numpy as np
import requests
import py3Dmol
from Bio.PDB import PDBParser
from Bio.Align import PairwiseAligner
from Bio.SeqUtils import seq1

response = requests.get(
    "https://rest.uniprot.org/uniprotkb/P23458.json", timeout=60
)
response.raise_for_status()
reference = response.json()["sequence"]["value"]

domains = [
    ("FERM", 34, 420, "green"),
    ("SH2", 439, 544, "cyan"),
    ("JH2 pseudokinase", 583, 855, "orange"),
    ("JH1 kinase", 875, 1153, "red"),
]

# Align the entire model sequence.
# Unmodeled ends of the reference are allowed, supporting Shan's fragment.
aligner = PairwiseAligner()
aligner.mode = "global"
aligner.match_score = 2
aligner.mismatch_score = -1
aligner.open_gap_score = -8
aligner.extend_gap_score = -0.5
aligner.query_left_gap_score = 0
aligner.query_right_gap_score = 0

parser = PDBParser(QUIET=True)

view = py3Dmol.view(
    width=1100, height=850,
    viewergrid=(2, 2),
    linked=False,
)

for i, (filename, pdb_text) in enumerate(models):
    panel = (i // 2, i % 2)
    first_pdb = pdb_text.split("ENDMDL")[0]
    structure = parser.get_structure(filename, io.StringIO(first_pdb))

    view.addModel(first_pdb, "pdb", viewer=panel)
    view.setStyle(
        {}, {"cartoon": {"color": "lightgray"}}, viewer=panel
    )
    view.setBackgroundColor("white", viewer=panel)

    print(f"\n{filename}")

    for chain in structure[0]:
        residues = [
            r for r in chain
            if "CA" in r and (r.id[0] == " " or r.resname == "MSE")
        ]
        if not residues:
            continue

        sequence = "".join(
            seq1(r.resname, custom_map={"MSE": "M"}, undef_code="X")
            for r in residues
        )

        alignment = aligner.align(reference, sequence)[0]
        mapping = {}

        for ref_block, pdb_block in zip(*alignment.aligned):
            for ref_index, pdb_index in zip(
                range(*ref_block), range(*pdb_block)
            ):
                mapping[pdb_index] = ref_index + 1

        identity = sum(
            sequence[j] == reference[pos - 1]
            for j, pos in mapping.items()
        ) / max(1, len(mapping))

        print(
            f"  Chain {chain.id}: {len(residues)} residues; "
            f"{len(mapping)} mapped; identity {identity:.1%}"
        )

        for name, start, end, color in domains:
            selected = [
                residues[j] for j, pos in mapping.items()
                if start <= pos <= end
            ]
            print(f"    {name}: {len(selected)} residues")

            if not selected:
                continue

            # Select directly by chain and PDB residue number.
            for residue in selected:
                selection = {
                    "chain": chain.id,
                    "resi": residue.id[1],
                }
                if residue.id[2].strip():
                    selection["icode"] = residue.id[2]

                view.setStyle(
                    selection,
                    {"cartoon": {"color": color}},
                    viewer=panel,
                )

            center = np.mean(
                [r["CA"].coord for r in selected], axis=0
            )
            view.addLabel(
                name,
                {
                    "position": {
                        "x": float(center[0]),
                        "y": float(center[1]),
                        "z": float(center[2]),
                    },
                    "fontSize": 14,
                    "fontColor": "black",
                    "backgroundColor": color,
                    "backgroundOpacity": 0.8,
                    "inFront": True,
                },
                viewer=panel,
            )

    view.zoomTo(viewer=panel)

for i, (name, _) in enumerate(models):
    print(f"Panel ({i // 2 + 1}, {i % 2 + 1}): {name}")

view.show()

/usr/local/lib/python3.13/dist-packages/Bio/Align/__init__.py:4412: BiopythonDeprecationWarning: The attribute 'query_left_gap_score' was renamed to 'left_deletion_score'. This was done to be consistent with the
AlignmentCounts object returned by the .counts method of an Alignment object.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/Bio/Align/__init__.py:4412: BiopythonDeprecationWarning: The attribute 'query_right_gap_score' was renamed to 'right_deletion_score'. This was done to be consistent with the
AlignmentCounts object returned by the .counts method of an Alignment object.
  warnings.warn(



Jeanpierre2025_Closed_WT_JAK1_best_model.pdb
  Chain A: 1154 residues; 1154 mapped; identity 100.0%
    FERM: 387 residues
    SH2: 106 residues
    JH2 pseudokinase: 273 residues
    JH1 kinase: 279 residues

Jeanpierre2025_Open_WT_JAK1_best_model.pdb
  Chain A: 1154 residues; 1154 mapped; identity 100.0%
    FERM: 387 residues
    SH2: 106 residues
    JH2 pseudokinase: 273 residues
    JH1 kinase: 279 residues

Shan2014_NIHMS704086-supplement-PDB_coordinates_for_JAK1_model.pdb
  Chain X: 592 residues; 592 mapped; identity 98.5%
    FERM: 0 residues
    SH2: 0 residues
    JH2 pseudokinase: 273 residues
    JH1 kinase: 279 residues

AlphaFold JAK1 — P23458
  Chain A: 1154 residues; 1154 mapped; identity 100.0%
    FERM: 387 residues
    SH2: 106 residues
    JH2 pseudokinase: 273 residues
    JH1 kinase: 279 residues
Panel (1, 1): Jeanpierre2025_Closed_WT_JAK1_best_model.pdb
Panel (1, 2): Jeanpierre2025_Open_WT_JAK1_best_model.pdb
Panel (2, 1): Shan2014_NIHMS704086-supplement-PDB_co

3Dmol.js failed to load for some reason. Please check your browser console for error messages.